# House Price Prediction — Decision Tree Regression
### Exploratory Data Analysis & Model Building

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/housing.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nTarget stats:\n', df['medv'].describe())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['medv'], bins=30, kde=True, color='#2ecc71', edgecolor='white', ax=axes[0])
axes[0].set_title('Target Distribution — Median House Value')
axes[0].set_xlabel('MEDV ($1000s)')
sns.boxplot(y=df['medv'], color='#27ae60', ax=axes[1])
axes[1].set_title('Box Plot — MEDV')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(13, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.4, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
corr_target = df.corr()['medv'].drop('medv').sort_values()
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_target]
plt.figure(figsize=(10, 6))
corr_target.plot(kind='barh', color=colors, edgecolor='black', linewidth=0.5)
plt.title('Feature Correlation with MEDV')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
top_features = corr_target.abs().sort_values(ascending=False).index[:4].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['medv'], alpha=0.4, color='#2ecc71', edgecolor='white', s=25)
    m, b = np.polyfit(df[feat], df['medv'], 1)
    xs = sorted(df[feat])
    axes[i].plot(xs, [m*x+b for x in xs], 'r--', lw=1.5)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MEDV')
    axes[i].set_title(f'{feat} vs MEDV')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
X = df.drop('medv', axis=1)
y = df['medv']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 4. Finding Optimal Tree Depth

In [ ]:
depths = range(1, 21)
train_r2, test_r2 = [], []
for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(X_train_sc, y_train)
    train_r2.append(r2_score(y_train, dt.predict(X_train_sc)))
    test_r2.append(r2_score(y_test,  dt.predict(X_test_sc)))
best_depth = depths[np.argmax(test_r2)]
print(f'Best Depth: {best_depth}  |  Best R²: {max(test_r2):.4f}')
plt.figure(figsize=(12, 5))
plt.plot(depths, train_r2, marker='o', label='Train R²', color='#3498db')
plt.plot(depths, test_r2,  marker='s', label='Test R²',  color='#2ecc71')
plt.axvline(best_depth, linestyle='--', color='gray', label=f'Best Depth={best_depth}')
plt.xlabel('Max Depth')
plt.ylabel('R² Score')
plt.title('R² vs Max Depth')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Train Final Model & Evaluate

In [ ]:
dt = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred = dt.predict(X_test_sc)
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_pred, alpha=0.55, color='#2ecc71', edgecolor='white', s=35)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Fit')
axes[0].set_xlabel('Actual MEDV')
axes[0].set_ylabel('Predicted MEDV')
axes[0].set_title(f'Actual vs Predicted  (R² = {r2:.3f})')
axes[0].legend()
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.55, color='#e74c3c', edgecolor='white', s=35)
axes[1].axhline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted MEDV')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#27ae60', edgecolor='black', linewidth=0.5)
plt.title('Feature Importances — Decision Tree')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(22, 8))
plot_tree(dt, feature_names=X.columns.tolist(), filled=True, rounded=True, fontsize=9, max_depth=3)
plt.title('Decision Tree Structure (max depth shown: 3)')
plt.tight_layout()
plt.show()

In [ ]:
cv_scores = cross_val_score(dt, scaler.transform(X), y, cv=10, scoring='r2')
print(f'10-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Save Model & Scaler

In [ ]:
joblib.dump(dt,     '../models/dt_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('dt_model.pkl and scaler.pkl saved to models/')